# Dependências

In [1]:
import pandas as pd               
import numpy as np                 
import os                         
import pandas_gbq                  
from google.cloud import bigquery  
import glob                        
import openpyxl                    
import csv                         
import re


c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [2]:
diretorio = 'G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\[PEP - GF] Painel estatístico de pessoal do governo federal'

In [3]:
os.listdir(diretorio)

['ingresso_GF.csv',
 'carreiras_vacancias_11_05.csv',
 'desligamento_GF.csv',
 'ingresso_GF_082023.csv',
 'pep25.xlsx',
 'pepingresso25.xlsx',
 'PEP_cargos_funcoes_das.xlsx',
 'PEP_2024_cargos_funcoes.xlsx',
 'PEP_2025_out_todos_cargos.xlsx',
 'PEP_2025_nov_cargos_funcoes.xlsx',
 '[Anuário 2025] Representatividade de gênero na burocracia.xlsx',
 'pep_teste_ministerio_genero.csv']

In [4]:
os.chdir(diretorio)  

In [5]:
df = pd.read_excel('[Anuário 2025] Representatividade de gênero na burocracia.xlsx', sheet_name='ministerio_genero')

In [6]:
df

,orgao,area,feminino,masculino,percentual
0,Ministério da Justiça e Segurança Pública,"Defesa, segurança e justiça",5253,25494,0.170846
1,Ministério de Portos e Aeroportos,"Infraestrutura, desenvolvimento econômico e me...",539,1488,0.265910
2,Vice-Presidência Da República,"Economia, gestão e planejamento",21,55,0.276316
3,Ministério do Meio Ambiente e Mudança do Clima,"Infraestrutura, desenvolvimento econômico e me...",3438,8618,0.285169
4,Presidência Da Republica,"Economia, gestão e planejamento",1395,3408,0.290443
5,Ministério Dos Transportes,"Infraestrutura, desenvolvimento econômico e me...",992,2397,0.292712
6,Ministério De Minas E Energia,"Infraestrutura, desenvolvimento econômico e me...",1058,2339,0.311451
7,Ministério do Desenvolvimento Agrário e Agricu...,"Infraestrutura, desenvolvimento econômico e me...",1176,2335,0.334947
8,Controladoria-Geral da União,Regulatória e controle,828,1635,0.336175
9,"Ministério da Ciência, Tecnologia e Inovação","Infraestrutura, desenvolvimento econômico e me...",2138,4186,0.338077


In [ ]:
df1 = df.melt(
    id_vars=['orgao', 'area', 'percentual'],
    value_vars=['feminino', 'masculino'],
    var_name='genero',
    value_name='quantidade'
)
df1

,orgao,area,percentual,genero,quantidade
0,Ministério da Justiça e Segurança Pública,"Defesa, segurança e justiça",0.170846,feminino,5253
1,Ministério de Portos e Aeroportos,"Infraestrutura, desenvolvimento econômico e me...",0.265910,feminino,539
2,Vice-Presidência Da República,"Economia, gestão e planejamento",0.276316,feminino,21
3,Ministério do Meio Ambiente e Mudança do Clima,"Infraestrutura, desenvolvimento econômico e me...",0.285169,feminino,3438
4,Presidência Da Republica,"Economia, gestão e planejamento",0.290443,feminino,1395
...,...,...,...,...,...
67,Defensoria Pública da União,"Defesa, segurança e justiça",0.562594,masculino,290
68,Ministério do Desenvolvimento e Assistência So...,Social,0.569395,masculino,242
69,Ministérios dos Direitos Humanos e da Cidadania,Social,0.580756,masculino,122
70,Ministério da Igualdade Racial,Social,0.656934,masculino,47


In [7]:
df['proporcao_fem'] = (df['feminino']/(df['feminino']+df['masculino'])*100).round(2)
df['proporcao_masc'] = (df['masculino']/(df['feminino']+df['masculino'])*100).round(2)
df

,orgao,area,feminino,masculino,percentual,proporcao_fem,proporcao_masc
0,Ministério da Justiça e Segurança Pública,"Defesa, segurança e justiça",5253,25494,0.170846,17.08,82.92
1,Ministério de Portos e Aeroportos,"Infraestrutura, desenvolvimento econômico e me...",539,1488,0.265910,26.59,73.41
2,Vice-Presidência Da República,"Economia, gestão e planejamento",21,55,0.276316,27.63,72.37
3,Ministério do Meio Ambiente e Mudança do Clima,"Infraestrutura, desenvolvimento econômico e me...",3438,8618,0.285169,28.52,71.48
4,Presidência Da Republica,"Economia, gestão e planejamento",1395,3408,0.290443,29.04,70.96
5,Ministério Dos Transportes,"Infraestrutura, desenvolvimento econômico e me...",992,2397,0.292712,29.27,70.73
6,Ministério De Minas E Energia,"Infraestrutura, desenvolvimento econômico e me...",1058,2339,0.311451,31.15,68.85
7,Ministério do Desenvolvimento Agrário e Agricu...,"Infraestrutura, desenvolvimento econômico e me...",1176,2335,0.334947,33.49,66.51
8,Controladoria-Geral da União,Regulatória e controle,828,1635,0.336175,33.62,66.38
9,"Ministério da Ciência, Tecnologia e Inovação","Infraestrutura, desenvolvimento econômico e me...",2138,4186,0.338077,33.81,66.19


In [12]:
df2 = df1.melt(
    id_vars=['orgao', 'area', 'genero', 'quantidade'],
    value_vars=['proporcao_fem', 'proporcao_masc'],
    var_name='prop_genero',
    value_name='quantidade'
)
df2

ValueError: value_name (quantidade) cannot match an element in the DataFrame columns.

In [10]:
df1['genero'] = df1['genero'].str.title()
df1['area'] = df1['area'].str.title()
df1['orgao'] = df1['orgao'].str.title()
df1

tipo,orgao,area,genero,prop_genero,quantidade
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Fem,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masc,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Fem,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masc,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Fem,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masc,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Fem,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masc,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Fem,27.63,21.0


In [11]:
def transforme(nome):
    nome =re.sub(r"\bFem\b", "Feminino", nome)
    nome =re.sub(r"\bMasc\b", "Masculino", nome)

    return nome

In [12]:
df1['genero'] = df1['genero'].apply(transforme)
df1

tipo,orgao,area,genero,prop_genero,quantidade
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21.0


In [13]:
df1 = df1.rename(columns={'quantidade':'quantidade_vinculos'})

In [14]:
df1 = df1[['orgao', 'area', 'genero', 'prop_genero', 'quantidade_vinculos']]
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21.0


In [16]:
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21.0


In [ ]:
df1['area'].unique()

array(['Advocacia-Geral Da União', 'Controladoria-Geral Da União',
       'Defensoria Pública Da União', 'Ministério Das Mulheres',
       'Ministério De Minas E Energia', 'Ministério Do Turismo',
       'Ministério Dos Transportes',
       'Ministério Gestão E Inovação Em Serviços Públicos',
       'Ministério Da Agricultura E Pecuária',
       'Ministério Da Ciência, Tecnologia E Inovação',
       'Ministério Da Cultura', 'Ministério Da Defesa',
       'Ministério Da Educação', 'Ministério Da Fazenda',
       'Ministério Da Igualdade Racial',
       'Ministério Da Integração E Do Desenvolvimento Regional',
       'Ministério Da Justiça E Segurança Pública',
       'Ministério Da Pesca E Aquicultura',
       'Ministério Da Previdência Social', 'Ministério Da Saúde',
       'Ministério Das Cidades', 'Ministério Das Comunicações',
       'Ministério Das Relações Exteriores',
       'Ministério De Portos E Aeroportos',
       'Ministério Do Desenvolvimento Agrário E Agricultura Familia',

In [ ]:
df1['genero'].unique().tolist()

In [26]:
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21.0


In [27]:
df1['quantidade_vinculos'] = df1['quantidade_vinculos'].astype(int)
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21


# Upload

In [20]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   orgao                72 non-null     object 
 1   area                 72 non-null     object 
 2   genero               72 non-null     object 
 3   prop_genero          72 non-null     float64
 4   quantidade_vinculos  72 non-null     float64
dtypes: float64(2), object(3)
memory usage: 2.9+ KB


In [24]:
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.82,4550.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.18,5601.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,33.62,828.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,66.38,1635.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.26,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,41.92,122.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.04,1395.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.96,3408.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,27.63,21.0


In [25]:
df1.to_csv('pep_teste_ministerio_genero.csv')

In [28]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [29]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   orgao                72 non-null     object 
 1   area                 72 non-null     object 
 2   genero               72 non-null     object 
 3   prop_genero          72 non-null     float64
 4   quantidade_vinculos  72 non-null     int32  
dtypes: float64(1), int32(1), object(3)
memory usage: 2.7+ KB


In [30]:
schema = [bigquery.SchemaField('orgao', 'STRING', description= 'Órgão'),
          bigquery.SchemaField('area', 'STRING', description= 'Agrupamento de áreas do órgão'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop_genero', 'FLOAT', description= 'Proporção de vínculos por genero'),
          ]

dataset_ref = client.dataset('perfil_remuneracao')

table_ref = dataset_ref.table('PEP_ministerio_genero') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df1, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=6190088b-26be-422f-a911-f540d8fd992f>